In [23]:
import pandas as pd
import os

csv_path = "../../../../../results/results.csv"
latex_out = "tables/vrp_results.tex"
df = pd.read_csv(csv_path)

In [24]:
def agent_type_name(t):
    t = str(t).upper()
    if t in ["GROUND", "G"]:
        return "G"
    elif t in ["AERIAL", "A"]:
        return "A"
    else:
        return t

def method_type_name(t):
    t = str(t)
    if "het" in t:
        return "Het"
    elif "hom" in t:
        return "Hom"
    else:
        return t

In [25]:
df["agent_type_norm"] = df["agent_type"].apply(agent_type_name)
df["method_norm"] = df["method"].apply(method_type_name)

In [26]:
# Count how many of each agent type per experiment
agent_count = {}
# Choose one baseline to avoid doubles
baseline = df["method_norm"].unique()[0]
for exp_id, agent_type, method in zip(df["exp_id"], df["agent_type_norm"], df["method_norm"]):
    if exp_id not in agent_count:
        agent_count[exp_id] = ""
    if  method == baseline:
        agent_count[exp_id] += f"{agent_type},"

# Add agent_config to each experiment
df["agent_config"] = [agent_count[exp_id] for exp_id in df["exp_id"]]

In [ ]:
agg_cols = ["total_cost", "distance", "time", "traversability", "collision", "safety"]

# Number of significant decimals for each metric
decimals_avg_max = [0, 0, 0, 3, 3, 3]  # for mean, min, max
decimals_std = [0, 0, 0, 2, 2, 2]      # for std

# Aggregate mean, std, min, max
summary = (
    df.groupby(["map", "method_norm", "agent_config"])[agg_cols]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

# Flatten MultiIndex columns
summary.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in summary.columns.values]

# --- Round numeric columns per metric ---
for i, m in enumerate(agg_cols):
    for suffix, dec in zip(["mean", "std", "min", "max"], [decimals_avg_max[i], decimals_std[i], decimals_avg_max[i], decimals_avg_max[i]]):
        col_name = f"{m}_{suffix}"
        if dec == 0:
            summary[col_name] = summary[col_name].round(0).astype(int)
        else:
            summary[col_name] = summary[col_name].round(dec)

# --- Create LaTeX-formatted mean ± std strings ---
for i, m in enumerate(agg_cols):
    summary[f"{m}_latex"] = summary.apply(
        lambda row: f"${row[f'{m}_mean']} \\pm {row[f'{m}_std']} ({row[f'{m}_max']})$",
        axis=1
    )

# --- Keep only relevant columns ---
cols = ["map", "method_norm", "agent_config"] + [f"{m}_latex" for m in agg_cols]
df_disp = summary[cols].copy()

# --- Sort for consistent order ---
df_disp = df_disp.sort_values(["map", "method_norm", "agent_config"]).reset_index(drop=True)

# --- Produce LaTeX-friendly row outputs per metric ---
for metric in agg_cols:
    print(f"\n% ===== {metric} =====")
    
    # Method order (reversed from appearance)
    method_order = df_disp["method_norm"].unique()[::-1]
    
    pivot = (
        df_disp.pivot_table(
            index=["method_norm"],
            columns=["map", "agent_config"],
            values=f"{metric}_latex",
            aggfunc="first"
        )
        .reindex(method_order)
    )
    
    # Print LaTeX rows
    for method, row in pivot.iterrows():
        values = " & ".join(row.fillna("").tolist())
        print(f"{metric} & {method} & {values} \\\\")



% ===== total_cost =====
total_cost & Het & $31686.0 \pm 15580.0 (39496)$ & $46377.0 \pm 4.0 (46380)$ & $26380.0 \pm 4203.0 (28918)$ & $36152.0 \pm 3885.0 (40349)$ & $43226.0 \pm 174.0 (43350)$ & $53119.0 \pm 214.0 (53270)$ & $28176.0 \pm 9775.0 (33082)$ & $41794.0 \pm 2666.0 (45263)$ \\
total_cost & Hom & $19728.0 \pm 11.0 (19740)$ & $32042.0 \pm 0.0 (32042)$ & $20061.0 \pm 717.0 (20516)$ & $26534.0 \pm 231.0 (26745)$ & $20492.0 \pm 128.0 (20587)$ & $33550.0 \pm 35.0 (33574)$ & $16670.0 \pm 163.0 (16762)$ & $26086.0 \pm 134.0 (26208)$ \\

% ===== distance =====
distance & Het & $944.0 \pm 633.0 (1666.0)$ & $1479.0 \pm 680.0 (1960.0)$ & $57.0 \pm 31.0 (96.0)$ & $92.0 \pm 45.0 (135.0)$ & $57.0 \pm 12.0 (75.0)$ & $94.0 \pm 42.0 (123.0)$ & $109.0 \pm 61.0 (159.0)$ & $176.0 \pm 63.0 (236.0)$ \\
distance & Hom & $821.0 \pm 23.0 (854.0)$ & $1382.0 \pm 7.0 (1387.0)$ & $61.0 \pm 6.0 (65.0)$ & $87.0 \pm 1.0 (88.0)$ & $49.0 \pm 2.0 (51.0)$ & $85.0 \pm 0.0 (85.0)$ & $94.0 \pm 2.0 (96.0)$ & $153.